# TSVD zadanie - Apache Spark

Tento notebook obsahuje cele riesenie zadania v jednom subore. Kod je pisany jednoduchsie, aby bolo vidiet jednotlive kroky:

1. nacitanie a priprava dat,
2. integracia poruch s meraniami,
3. sampling,
4. predspracovanie,
5. tvorba dennych priebehov,
6. train/test rozdelenie,
7. korelacie a vyber atributov,
8. trenovanie a vyhodnotenie klasifikatorov.

Poznamka: povodny `priebehy.parquet` ma cas ulozeny ako nanosekundovy timestamp. Spark 3.5 ho na Windowse nevie citat priamo, preto sa na zaciatku vytvori lokalna kopia v `prepared/priebehy_spark.parquet` s mikrosekundovym timestampom. Tato technicka priprava nemeni hodnoty merani.

In [1]:
from pathlib import Path
import os
import math
import shutil

import jdk4py
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier, DecisionTreeClassifier
from pyspark.ml.feature import Imputer, StringIndexer, VectorAssembler
from pyspark.ml.stat import Correlation

PROJECT_DIR = Path(r"C:\TSVD_zadanie")
DATASET_DIR = PROJECT_DIR / "dataset"
PREPARED_DIR = PROJECT_DIR / "prepared"

RAW_PARQUET = DATASET_DIR / "priebehy.parquet"
FAULTS_XLSX = DATASET_DIR / "Poruchy.xlsx"
SPARK_PARQUET = PREPARED_DIR / "priebehy_spark.parquet"

RANDOM_SEED = 42
NUMERIC_COLS = ["u1_norm", "u2_norm", "u3_norm", "i1_norm", "i2_norm", "i3_norm"]
FAULT_TYPES = [1, 2, 3, 4, 5, 6, 7, 8, 9, 13, 14, 15]

## 1. Technicka priprava vstupu

In [2]:
def prepare_parquet_for_spark(source=RAW_PARQUET, target=SPARK_PARQUET):
    """Preulozi t_utc z timestamp[ns] na timestamp[us], aby ho vedel citat Spark 3.5."""
    if target.exists():
        print(f"Pouzivam existujuci pripraveny subor: {target}")
        return target

    target.parent.mkdir(parents=True, exist_ok=True)
    parquet_file = pq.ParquetFile(source)
    writer = None

    try:
        for row_group in range(parquet_file.num_row_groups):
            table = parquet_file.read_row_group(row_group)
            fields = []
            for field in table.schema:
                if field.name == "t_utc":
                    fields.append(pa.field("t_utc", pa.timestamp("us", tz="UTC")))
                else:
                    fields.append(field)
            table = table.cast(pa.schema(fields))

            if writer is None:
                writer = pq.ParquetWriter(target, table.schema, compression="snappy")
            writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()

    print(f"Vytvoreny Spark Parquet: {target}")
    return target

prepare_parquet_for_spark()

Vytvoreny Spark Parquet: C:\TSVD_zadanie\prepared\priebehy_spark.parquet


WindowsPath('C:/TSVD_zadanie/prepared/priebehy_spark.parquet')

## 2. Spark session

In [3]:
os.environ["JAVA_HOME"] = str(jdk4py.JAVA_HOME)
os.environ["PYSPARK_PYTHON"] = str(PROJECT_DIR / ".venv" / "Scripts" / "python.exe")
os.environ["PYSPARK_DRIVER_PYTHON"] = os.environ["PYSPARK_PYTHON"]

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("TSVD zadanie")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "32")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## 3. Nacitanie merani a poruch

In [4]:
measurements = (
    spark.read.parquet(str(SPARK_PARQUET))
    .drop("elektromer")
    .withColumn("t_utc", F.col("t_utc").cast("timestamp"))
)

print("Pocet merani:", measurements.count())
measurements.printSchema()
measurements.show(5, truncate=False)

Pocet merani: 7659371
root
 |-- t_utc: timestamp (nullable = true)
 |-- u2_norm: double (nullable = true)
 |-- i3_norm: double (nullable = true)
 |-- i2_norm: double (nullable = true)
 |-- i1_norm: double (nullable = true)
 |-- u1_norm: double (nullable = true)
 |-- u3_norm: double (nullable = true)
 |-- eic: string (nullable = true)



+-------------------+-------+-------+-------+-------+-------+-------+----------------+
|t_utc              |u2_norm|i3_norm|i2_norm|i1_norm|u1_norm|u3_norm|eic             |
+-------------------+-------+-------+-------+-------+-------+-------+----------------+
|2022-12-16 23:10:00|14123.0|0.0    |0.0    |0.0    |12703.0|12814.0|24ZVS0000003196R|
|2022-12-16 23:20:00|14068.0|0.0    |0.0    |0.0    |12658.0|12747.0|24ZVS0000003196R|
|2022-12-16 23:30:00|14087.0|0.0    |0.0    |0.0    |12634.0|12758.0|24ZVS0000003196R|
|2022-12-16 23:40:00|14179.0|0.0    |0.0    |0.0    |12632.0|12863.0|24ZVS0000003196R|
|2022-12-16 23:50:00|14174.0|0.0    |0.0    |0.0    |12630.0|12848.0|24ZVS0000003196R|
+-------------------+-------+-------+-------+-------+-------+-------+----------------+
only showing top 5 rows



In [5]:
faults_pd = pd.read_excel(FAULTS_XLSX, sheet_name="Evidencia poruch")
fault_types_pd = pd.read_excel(FAULTS_XLSX, sheet_name="Typy poruch")

# Excel ma slovenske nazvy stlpcov s diakritikou. Aby notebook nebol citlivy na kodovanie,
# premenujeme prve 4 stlpce podla poradia: EIC, zaciatok, koniec, typ poruchy.
faults_pd = faults_pd.rename(columns={
    faults_pd.columns[0]: "eic",
    faults_pd.columns[1]: "fault_start",
    faults_pd.columns[2]: "fault_end",
    faults_pd.columns[3]: "fault_type",
})

faults_pd = faults_pd.dropna(subset=["eic", "fault_type"])
faults_pd["fault_start"] = pd.to_datetime(faults_pd["fault_start"])
faults_pd["fault_end"] = pd.to_datetime(faults_pd["fault_end"])
faults_pd["fault_type"] = faults_pd["fault_type"].astype(int)

faults = spark.createDataFrame(faults_pd[["eic", "fault_start", "fault_end", "fault_type"]])
faults.show(10, truncate=False)

fault_types_pd

+----------------+-------------------+-------------------+----------+
|eic             |fault_start        |fault_end          |fault_type|
+----------------+-------------------+-------------------+----------+
|24ZVS0000003196R|2023-01-16 23:00:00|2023-06-06 22:00:00|4         |
|24ZVS0000001976B|2022-12-22 23:00:00|2023-03-13 23:00:00|5         |
|24ZVS0000002341C|2023-01-25 23:00:00|2024-07-24 22:00:00|4         |
|24ZVS0000003211K|NULL               |2023-10-22 22:00:00|3         |
|24ZVS0000639518F|2023-03-18 23:00:00|2024-08-21 22:00:00|3         |
|24ZVS0000038467G|NULL               |2023-06-26 22:00:00|1         |
|24ZVS0000074609I|NULL               |2023-06-23 22:00:00|1         |
|24ZVS0000002475S|2023-09-04 22:00:00|2023-09-07 22:00:00|4         |
|24ZVS00000020943|2023-10-05 22:00:00|2023-11-05 23:00:00|4         |
|24ZVS0000018647Q|2023-07-24 22:00:00|2023-11-21 23:00:00|4         |
+----------------+-------------------+-------------------+----------+
only showing top 10 

,Číselník,Popis poruchy
0,1,Chybný PTP vo fáze L1
1,2,Chybný PTP vo fáze L2
2,3,Chybný PTP vo fáze L3
3,4,Výpadok napätia vo fáze L1
4,5,Výpadok napätia vo fáze L2
5,6,Výpadok napätia vo fáze L3
6,7,Porucha P34 vo fáze L1
7,8,Porucha P34 vo fáze L2
8,9,Porucha P34 vo fáze L3
9,13,Vykratený prúd vo fáze L1


## 4. Integracia poruch s meraniami

Poruchy su intervaly. Meranie je oznacene ako poruchove vtedy, ked:

- ma rovnake `eic`,
- cas merania je medzi zaciatkom a koncom poruchy.

Ak ma jedno odberne miesto naraz viac poruch, ulozi sa ich mnozina, napriklad `1+4`. Pre binarnu klasifikaciu pouzijeme `target_binary`, pre multiclass klasifikaciu pouzijeme `target_multiclass`.

In [6]:
bounds = measurements.agg(F.min("t_utc").alias("min_t"), F.max("t_utc").alias("max_t"))

fault_intervals = (
    faults.crossJoin(bounds)
    .withColumn("start_t", F.coalesce(F.col("fault_start"), F.col("min_t")))
    .withColumn("end_t", F.coalesce(F.to_timestamp(F.date_add(F.to_date("fault_end"), 1)), F.col("max_t") + F.expr("INTERVAL 1 SECOND")))
    .select("eic", "start_t", "end_t", "fault_type")
)

m = measurements.withColumn("measurement_id", F.monotonically_increasing_id())

joined = m.join(
    F.broadcast(fault_intervals),
    (m.eic == fault_intervals.eic) & (m.t_utc >= fault_intervals.start_t) & (m.t_utc < fault_intervals.end_t),
    "left"
)

labels = (
    joined.groupBy("measurement_id")
    .agg(
        F.array_sort(F.collect_set("fault_type")).alias("fault_types"),
        F.max(F.when(F.col("fault_type").isNotNull(), 1).otherwise(0)).alias("target_binary")
    )
    .withColumn("fault_types", F.expr("filter(fault_types, x -> x is not null)"))
    .withColumn("target_multiclass", F.when(F.size("fault_types") == 0, "0").otherwise(F.concat_ws("+", F.col("fault_types"))))
)

integrated = m.join(labels, "measurement_id").drop("measurement_id").cache()

integrated.groupBy("target_binary", "target_multiclass").count().orderBy("target_binary", "target_multiclass").show(30, truncate=False)

+-------------+-----------------+-------+
|target_binary|target_multiclass|count  |
+-------------+-----------------+-------+
|0            |0                |3233043|
|1            |1                |1554590|
|1            |13               |32850  |
|1            |2                |586212 |
|1            |3                |1323365|
|1            |4                |266708 |
|1            |5                |94620  |
|1            |6                |68899  |
|1            |7                |222251 |
|1            |8                |194591 |
|1            |9                |82242  |
+-------------+-----------------+-------+



## 5. Stratifikovany sampling 10 %

In [7]:
fractions = {row["target_binary"]: 0.10 for row in integrated.select("target_binary").distinct().collect()}
sample_10 = integrated.sampleBy("target_binary", fractions=fractions, seed=RANDOM_SEED).cache()

print("Pocet riadkov vo vzorke:", sample_10.count())
sample_10.groupBy("target_binary", "target_multiclass").count().orderBy("target_binary", "target_multiclass").show(30, truncate=False)

Pocet riadkov vo vzorke: 764928


+-------------+-----------------+------+
|target_binary|target_multiclass|count |
+-------------+-----------------+------+
|0            |0                |322457|
|1            |1                |155202|
|1            |13               |3333  |
|1            |2                |58509 |
|1            |3                |132345|
|1            |4                |26711 |
|1            |5                |9447  |
|1            |6                |6824  |
|1            |7                |22376 |
|1            |8                |19549 |
|1            |9                |8175  |
+-------------+-----------------+------+



## 6. Chybajuce hodnoty, popisne charakteristiky a outliery

In [8]:
missing = integrated.agg(*[
    F.count(F.when(F.col(c).isNull() | F.isnan(c), c)).alias(c)
    for c in NUMERIC_COLS
])
missing.show()

integrated.select(NUMERIC_COLS).summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max").show(truncate=False)

integrated.agg(
    *[F.skewness(c).alias(f"{c}_skewness") for c in NUMERIC_COLS],
    *[F.kurtosis(c).alias(f"{c}_kurtosis") for c in NUMERIC_COLS]
).show(truncate=False)

+-------+-------+-------+-------+-------+-------+
|u1_norm|u2_norm|u3_norm|i1_norm|i2_norm|i3_norm|
+-------+-------+-------+-------+-------+-------+
|  77638|  77622|  77638|  77638|  77622|  77638|
+-------+-------+-------+-------+-------+-------+



+-------+------------------+-----------------+------------------+------------------+------------------+------------------+
|summary|u1_norm           |u2_norm          |u3_norm           |i1_norm           |i2_norm           |i3_norm           |
+-------+------------------+-----------------+------------------+------------------+------------------+------------------+
|count  |7581733           |7581749          |7581733           |7581733           |7581749           |7581733           |
|mean   |736.7045705509078 |738.5095031924687|763.7825256472858 |11.359328966246457|11.923238116111174|12.04502518686281 |
|stddev |2561.4779262403267|2505.182530877948|2538.8969524941244|26.63775450154358 |28.21802047901344 |28.402429391343123|
|min    |0.0               |0.0              |0.0               |0.0               |0.0               |0.0               |
|25%    |236.23            |236.24           |236.63            |0.0               |0.2               |0.0               |
|50%    |238.77 

+-----------------+----------------+-----------------+-----------------+-----------------+-----------------+-----------------+------------------+-----------------+-----------------+-----------------+-----------------+
|u1_norm_skewness |u2_norm_skewness|u3_norm_skewness |i1_norm_skewness |i2_norm_skewness |i3_norm_skewness |u1_norm_kurtosis |u2_norm_kurtosis  |u3_norm_kurtosis |i1_norm_kurtosis |i2_norm_kurtosis |i3_norm_kurtosis |
+-----------------+----------------+-----------------+-----------------+-----------------+-----------------+-----------------+------------------+-----------------+-----------------+-----------------+-----------------+
|4.913942735064321|4.78455475451433|4.592119012369929|5.878188011875002|6.180364550089781|5.952949727495034|22.54985500250665|20.979333034885805|19.16203220634391|55.45366886312619|63.40983016134621|61.35374219472189|
+-----------------+----------------+-----------------+-----------------+-----------------+-----------------+-----------------+--

In [9]:
def winsorize_iqr(df, cols):
    """Hodnoty mimo IQR hranic nahradi dolnou alebo hornou hranicou."""
    result = df
    bounds = []
    for c in cols:
        q1, q3 = df.approxQuantile(c, [0.25, 0.75], 0.01)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        bounds.append((c, q1, q3, lower, upper))
        result = result.withColumn(c, F.when(F.col(c) < lower, lower).when(F.col(c) > upper, upper).otherwise(F.col(c)))
    return result, bounds

no_outliers, outlier_bounds = winsorize_iqr(integrated, NUMERIC_COLS)

spark.createDataFrame(outlier_bounds, ["atribut", "q1", "q3", "dolna_hranica", "horna_hranica"]).show(truncate=False)

imputer = Imputer(strategy="median", inputCols=NUMERIC_COLS, outputCols=[c + "_imp" for c in NUMERIC_COLS])
imputed = imputer.fit(no_outliers).transform(no_outliers)

for c in NUMERIC_COLS:
    imputed = imputed.drop(c).withColumnRenamed(c + "_imp", c)

cleaned = imputed.cache()

+-------+------------------+------------------+-------------------+------------------+
|atribut|q1                |q3                |dolna_hranica      |horna_hranica     |
+-------+------------------+------------------+-------------------+------------------+
|u1_norm|236.11            |241.0             |228.77500000000003 |248.33499999999998|
|u2_norm|236.20000000000002|241.04            |228.94000000000005 |248.29999999999995|
|u3_norm|236.67000000000002|241.0             |230.17500000000004 |247.49499999999998|
|i1_norm|0.0               |10.4              |-15.600000000000001|26.0              |
|i2_norm|0.176             |10.200000000000001|-14.860000000000001|25.236000000000004|
|i3_norm|0.0               |10.200000000000001|-15.3              |25.5              |
+-------+------------------+------------------+-------------------+------------------+



## 7. Transformacia na denne priebehy

Merania su v 10-minutovych intervaloch, preto jeden cely den obsahuje 144 merani. Den oznacime ako poruchovy, ak sa pocas dna vyskytla aspon jedna porucha.

In [10]:
agg_exprs = []
for c in NUMERIC_COLS:
    agg_exprs += [
        F.avg(c).alias(f"{c}_avg"),
        F.max(c).alias(f"{c}_max"),
        F.min(c).alias(f"{c}_min"),
        F.skewness(c).alias(f"{c}_skewness"),
        F.kurtosis(c).alias(f"{c}_kurtosis"),
    ]

daily = (
    cleaned.withColumn("day", F.to_date("t_utc"))
    .groupBy("eic", "day")
    .agg(
        F.count("*").alias("n_samples"),
        F.max("target_binary").alias("target_binary"),
        F.array_sort(F.array_distinct(F.flatten(F.collect_list("fault_types")))).alias("daily_fault_types"),
        *agg_exprs
    )
    .where(F.col("n_samples") >= 144)
    .withColumn("target_multiclass", F.when(F.size("daily_fault_types") == 0, "0").otherwise(F.concat_ws("+", F.col("daily_fault_types"))))
    .drop("daily_fault_types")
    .cache()
)

print("Pocet dennych priebehov:", daily.count())
daily.groupBy("target_binary", "target_multiclass").count().orderBy("target_binary", "target_multiclass").show(50, truncate=False)

Pocet dennych priebehov: 52449


+-------------+-----------------+-----+
|target_binary|target_multiclass|count|
+-------------+-----------------+-----+
|0            |0                |21661|
|1            |1                |10793|
|1            |13               |230  |
|1            |2                |4071 |
|1            |3                |9190 |
|1            |4                |1870 |
|1            |5                |674  |
|1            |6                |484  |
|1            |7                |1549 |
|1            |8                |1354 |
|1            |9                |573  |
+-------------+-----------------+-----+



## 8. Train/test rozdelenie

In [11]:
w = Window.partitionBy("target_binary").orderBy(F.rand(RANDOM_SEED))
class_counts = daily.groupBy("target_binary").count().withColumnRenamed("count", "class_count")

ranked = (
    daily.join(class_counts, "target_binary")
    .withColumn("rn", F.row_number().over(w))
    .withColumn("is_train", F.col("rn") <= F.col("class_count") * 0.8)
)

train = ranked.where("is_train").drop("class_count", "rn", "is_train").cache()
test = ranked.where(~F.col("is_train")).drop("class_count", "rn", "is_train").cache()

print("Train:", train.count())
print("Test:", test.count())

Train: 41958


Test: 10491


## 9. Korelacie a vyber atributov

In [12]:
feature_cols = [
    c for c, t in daily.dtypes
    if c not in ["eic", "day", "target_binary", "target_multiclass"] and t in ["double", "int", "bigint"]
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_for_corr", handleInvalid="skip")
corr_df = assembler.transform(train)
corr_matrix = Correlation.corr(corr_df, "features_for_corr", "pearson").head()[0]

print("Pocet atributov pre korelacie:", len(feature_cols))
print(corr_matrix)

Pocet atributov pre korelacie: 31
DenseMatrix([[ 1.00000000e+00,  1.38058669e-02,  8.12985765e-02,
              -1.56558794e-02,  3.68645457e-02, -1.64520273e-02,
               4.22064407e-02,  1.26245405e-01,  3.70085083e-03,
               3.87530331e-02, -1.85755503e-02,  2.00214215e-02,
               5.96827150e-02, -2.19401487e-03,  2.32603300e-02,
              -1.55620010e-02, -5.11887534e-02,  3.39000470e-03,
              -9.91848348e-02, -7.03306138e-02, -2.22409835e-02,
              -3.64373572e-02,  1.52242613e-02, -8.92807568e-02,
              -6.27159436e-02, -5.39013150e-02,  1.73684241e-01,
               1.73533386e-01, -7.23181122e-03, -9.50610588e-02,
              -4.07846344e-02],
             [ 1.38058669e-02,  1.00000000e+00,  9.45560308e-01,
               9.30309113e-01, -3.70020213e-01,  1.34306228e-01,
               9.19072715e-01,  8.68970036e-01,  8.57297394e-01,
              -3.43641089e-01,  1.98203590e-01,  9.22467004e-01,
               8.5637301

In [13]:
def add_class_weights(df, label_col):
    total = df.count()
    counts = df.groupBy(label_col).count()
    n_classes = counts.count()
    weights = counts.withColumn("class_weight", F.lit(total) / (F.lit(n_classes) * F.col("count"))).drop("count")
    return df.join(weights, label_col)


def feature_importance_rf(train_df, features, label_col, top_n=20):
    assembler = VectorAssembler(inputCols=features, outputCol="features", handleInvalid="skip")
    rf = RandomForestClassifier(labelCol=label_col, featuresCol="features", numTrees=80, seed=RANDOM_SEED)
    model = rf.fit(assembler.transform(train_df).select(label_col, "features"))
    importances = list(zip(features, model.featureImportances.toArray()))
    importances = sorted(importances, key=lambda x: float(x[1]), reverse=True)
    selected = [name for name, value in importances[:top_n] if value > 0]
    return selected, pd.DataFrame(importances, columns=["atribut", "dolezitost"])

selected_binary_features, binary_importance = feature_importance_rf(train, feature_cols, "target_binary", top_n=20)
selected_binary_features

['i2_norm_avg',
 'i1_norm_max',
 'i1_norm_avg',
 'i2_norm_max',
 'i3_norm_avg',
 'i3_norm_max',
 'i2_norm_min',
 'i3_norm_min',
 'i1_norm_min',
 'u2_norm_max',
 'u2_norm_avg',
 'u2_norm_min',
 'u3_norm_avg',
 'u1_norm_max',
 'i1_norm_skewness',
 'u1_norm_min',
 'u3_norm_max',
 'u3_norm_min',
 'u3_norm_skewness',
 'u1_norm_kurtosis']

In [14]:
binary_importance.head(20)

,atribut,dolezitost
0,i2_norm_avg,0.218843
1,i1_norm_max,0.147524
2,i1_norm_avg,0.129499
3,i2_norm_max,0.118236
4,i3_norm_avg,0.079767
5,i3_norm_max,0.051879
6,i2_norm_min,0.039282
7,i3_norm_min,0.032439
8,i1_norm_min,0.026075
9,u2_norm_max,0.018509


## 10. Vyhodnocovacie funkcie

In [15]:
def evaluate_predictions(predictions, label_col, prediction_col="prediction"):
    """Vypocita accuracy, weighted precision, weighted recall, weighted F1 a MCC z kontingencnej tabulky."""
    rows = predictions.groupBy(label_col, prediction_col).count().collect()
    labels = sorted({float(r[label_col]) for r in rows} | {float(r[prediction_col]) for r in rows})
    matrix = {(float(r[label_col]), float(r[prediction_col])): float(r["count"]) for r in rows}
    total = sum(matrix.values())

    if total == 0:
        return {"accuracy": 0, "precision": 0, "recall": 0, "f1": 0, "mcc": 0}

    trace = sum(matrix.get((x, x), 0.0) for x in labels)
    true_totals = {}
    pred_totals = {}
    weighted_precision = weighted_recall = weighted_f1 = 0.0

    for label in labels:
        tp = matrix.get((label, label), 0.0)
        true_total = sum(matrix.get((label, pred), 0.0) for pred in labels)
        pred_total = sum(matrix.get((true, label), 0.0) for true in labels)
        true_totals[label] = true_total
        pred_totals[label] = pred_total

        precision = tp / pred_total if pred_total else 0.0
        recall = tp / true_total if true_total else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

        weighted_precision += precision * true_total
        weighted_recall += recall * true_total
        weighted_f1 += f1 * true_total

    numerator = trace * total - sum(pred_totals[x] * true_totals[x] for x in labels)
    denominator_left = total**2 - sum(pred_totals[x] ** 2 for x in labels)
    denominator_right = total**2 - sum(true_totals[x] ** 2 for x in labels)
    denominator = math.sqrt(denominator_left * denominator_right) if denominator_left > 0 and denominator_right > 0 else 0
    mcc = numerator / denominator if denominator else 0

    return {
        "accuracy": trace / total,
        "precision": weighted_precision / total,
        "recall": weighted_recall / total,
        "f1": weighted_f1 / total,
        "mcc": mcc,
    }


def train_models(train_df, test_df, features, label_col, multiclass=False):
    train_weighted = add_class_weights(train_df, label_col)
    models = [
        ("logistic_regression", LogisticRegression(labelCol=label_col, featuresCol="features", weightCol="class_weight", maxIter=50, family="multinomial" if multiclass else "auto")),
        ("random_forest", RandomForestClassifier(labelCol=label_col, featuresCol="features", weightCol="class_weight", numTrees=100, maxDepth=8, seed=RANDOM_SEED)),
    ]

    if multiclass:
        models.append(("decision_tree", DecisionTreeClassifier(labelCol=label_col, featuresCol="features", weightCol="class_weight", maxDepth=8, seed=RANDOM_SEED)))
    else:
        models.append(("gradient_boosted_trees", GBTClassifier(labelCol=label_col, featuresCol="features", weightCol="class_weight", maxIter=50, maxDepth=5, seed=RANDOM_SEED)))

    results = []
    confusion_tables = {}

    for name, model in models:
        pipeline = Pipeline(stages=[VectorAssembler(inputCols=features, outputCol="features", handleInvalid="skip"), model])
        fitted = pipeline.fit(train_weighted)
        pred = fitted.transform(test_df).cache()
        metrics = evaluate_predictions(pred, label_col)
        results.append({"model": name, **metrics})
        confusion_tables[name] = pred.groupBy(label_col, "prediction").count().orderBy(label_col, "prediction").toPandas()

    return pd.DataFrame(results).sort_values(["f1", "mcc"], ascending=False), confusion_tables

## 11. Binarna klasifikacia - detekcia lubovolnej poruchy

In [16]:
binary_results, binary_confusions = train_models(
    train,
    test,
    selected_binary_features,
    label_col="target_binary",
    multiclass=False,
)

binary_results

,model,accuracy,precision,recall,f1,mcc
2,gradient_boosted_trees,0.976900,0.977439,0.976900,0.976906,0.954313
1,random_forest,0.970798,0.972181,0.970798,0.970803,0.942966
0,logistic_regression,0.832486,0.832516,0.832486,0.832497,0.664655


In [17]:
binary_confusions[binary_results.iloc[0]["model"]]

,target_binary,prediction,count
0,0,0.0,3304
1,0,1.0,23
2,1,0.0,136
3,1,1.0,3420


## 12. Multiclass klasifikacia - detekcia typov poruch

In [18]:
indexer = StringIndexer(inputCol="target_multiclass", outputCol="target_multiclass_index", handleInvalid="keep")
indexer_model = indexer.fit(daily)
daily_multi = indexer_model.transform(daily).cache()

ranked_multi = (
    daily_multi.join(class_counts, "target_binary")
    .withColumn("rn", F.row_number().over(w))
    .withColumn("is_train", F.col("rn") <= F.col("class_count") * 0.8)
)
train_multi = ranked_multi.where("is_train").drop("class_count", "rn", "is_train").cache()
test_multi = ranked_multi.where(~F.col("is_train")).drop("class_count", "rn", "is_train").cache()

multi_feature_cols = [
    c for c, t in daily_multi.dtypes
    if c not in ["eic", "day", "target_binary", "target_multiclass", "target_multiclass_index"] and t in ["double", "int", "bigint"]
]

selected_multi_features, multi_importance = feature_importance_rf(train_multi, multi_feature_cols, "target_multiclass_index", top_n=20)
selected_multi_features

['i2_norm_avg',
 'i1_norm_max',
 'i1_norm_avg',
 'i2_norm_max',
 'i3_norm_max',
 'i3_norm_avg',
 'i2_norm_min',
 'i3_norm_min',
 'u2_norm_avg',
 'u2_norm_max',
 'u3_norm_max',
 'u3_norm_avg',
 'i1_norm_min',
 'i1_norm_skewness',
 'u1_norm_max',
 'u1_norm_avg',
 'u1_norm_min',
 'u2_norm_min',
 'i1_norm_kurtosis',
 'u3_norm_min']

In [19]:
multi_importance.head(20)

,atribut,dolezitost
0,i2_norm_avg,0.167601
1,i1_norm_max,0.146584
2,i1_norm_avg,0.121907
3,i2_norm_max,0.118665
4,i3_norm_max,0.095347
5,i3_norm_avg,0.057477
6,i2_norm_min,0.039802
7,i3_norm_min,0.032753
8,u2_norm_avg,0.025904
9,u2_norm_max,0.024047


In [20]:
multiclass_results, multiclass_confusions = train_models(
    train_multi,
    test_multi,
    selected_multi_features,
    label_col="target_multiclass_index",
    multiclass=True,
)

multiclass_results

,model,accuracy,precision,recall,f1,mcc
1,random_forest,0.945251,0.955836,0.945251,0.948736,0.922379
2,decision_tree,0.867282,0.937797,0.867282,0.892660,0.826451
0,logistic_regression,0.732586,0.858951,0.732586,0.769125,0.667397


In [21]:
multiclass_confusions[multiclass_results.iloc[0]["model"]]

,target_multiclass_index,prediction,count
0,0.0,0.0,3422
1,0.0,1.0,2
2,0.0,2.0,43
3,0.0,3.0,23
4,0.0,4.0,4
5,0.0,5.0,4
6,0.0,6.0,7
7,0.0,7.0,133
8,0.0,8.0,10
9,0.0,9.0,1


## 13. Zaver

Podla overeneho behu v tomto datasete vysli najlepsie tieto modely:

- binarna detekcia poruchy: Gradient Boosted Trees,
- multiclass detekcia typu poruchy: Random Forest.

Pri odovzdani je vhodne do textu doplnit konkretne hodnoty metrik z tabuliek vyssie, pretoze sa mozu mierne zmenit podla verzie Spark/Python a nahodneho rozdelenia dat.

In [22]:
spark.stop()